# Nifty 100 Financial Intelligence Platform — Exploratory Analysis
This notebook demonstrates connecting to the project's SQLite database, running basic exploratory queries, computing financial ratios, and plotting key financial trends.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

db_path = '../data/nifty100.db'
conn = sqlite3.connect(db_path)
print("Connected to database successfully.")

## 1. Companies Summary
Let's pull a summary of sectors and companies in the database.

In [ ]:
companies = pd.read_sql("SELECT * FROM companies", conn)
print(f"Total companies: {len(companies)}")
companies.head()

In [ ]:
# Companies count by sector
plt.figure(figsize=(12, 6))
sns.countplot(data=companies, y='sector', order=companies['sector'].value_counts().index, palette='viridis')
plt.title('Distribution of Nifty 100 Companies by Sector')
plt.xlabel('Number of Companies')
plt.ylabel('Sector')
plt.show()

## 2. Historical Profit & Loss Trends
Let's look at the revenue and profit trend of a major company like RELIANCE.

In [ ]:
reliance_pl = pd.read_sql("""
    SELECT year, sales, net_profit, operating_profit
    FROM profitandloss
    WHERE company_id = 'RELIANCE'
    ORDER BY year
""", conn)

reliance_pl.head(10)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(reliance_pl['year'], reliance_pl['sales'], marker='o', label='Sales (Revenue)', color='darkblue')
plt.plot(reliance_pl['year'], reliance_pl['net_profit'], marker='s', label='Net Profit', color='teal')
plt.title('Reliance Industries — Sales & Net Profit Growth (Cr)')
plt.xlabel('Financial Year')
plt.ylabel('Rupees (Crore)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.show()

## 3. Financial Ratio Distribution
Let's examine the distribution of Return on Equity (ROE) across the universe.

In [ ]:
ratios = pd.read_sql("""
    SELECT r.*, c.sector
    FROM financial_ratios r
    JOIN companies c ON r.company_id = c.id
    WHERE r.year = (SELECT MAX(year) FROM financial_ratios)
""", conn)
ratios.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(ratios['return_on_equity_pct'].dropna(), bins=30, kde=True, color='teal')
plt.title('Return on Equity (ROE %) Distribution — Latest Year')
plt.xlabel('ROE %')
plt.ylabel('Count')
plt.show()

In [ ]:
conn.close()
print("Connection closed.")